# Oil & gas production decline curve

Pull the monthly production history for a producing well and plot the
volumes the source regulator publishes. This is the canonical
"decline curve" landmen and mineral owners look at to understand
lease performance.

### About the data

What gets reported per well varies by state regulator. Kansas (KCC)
publishes oil-only monthly volumes with years of history. North Dakota
(NDIC) publishes oil + associated gas but not produced water. Many
larger producing states (TX, CO, NM, WY) publish per-lease rather than
per-well. The notebook auto-picks the first well it finds with usable
history and plots whichever volumes the source provides — it does not
invent columns the source omits.

In [ ]:
%pip install --quiet 'sondio[geo]>=0.1.2,<0.2' matplotlib

import sondio
# sondio >= 0.1.2 reads your key from Colab Secrets (🔑 sidebar),
# Kaggle Secrets, SONDIO_API_KEY env var, or ~/.sondio/config — in that
# order. Set whichever fits your environment.
print(f"sondio {sondio.__version__}")

In [ ]:
# Find a producing well with enough monthly history to plot. We try
# known-good operator/state combos in order and pick the first well
# whose production series is >= 12 months long.
CANDIDATES = [
    {"state": "ND", "operator": "CONTINENTAL RESOURCES"},  # oil + associated gas
    {"state": "KS", "operator": "BEREXCO"},                # oil only, long history
]
well_row, prod = None, None
for q in CANDIDATES:
    wells = sondio.oilgas.wells(**q, limit=15)
    for _, candidate in wells.iterrows():
        p = sondio.oilgas.production(candidate["id"], months=180)
        if len(p) >= 12:
            well_row, prod = candidate, p.sort_values("production_date").reset_index(drop=True)
            break
    if well_row is not None:
        break

assert well_row is not None, "No candidate wells had >= 12 months of production."
print(f"{well_row['well_name']} — {well_row['operator_name']} ({well_row['external_id']}, {well_row.get('subdivision_code') or well_row.get('subdivision')})")
print(f"{len(prod)} months of production, earliest {prod['production_date'].min().date()}")
prod.head()

In [ ]:
# Decline curve. Plot only the volumes this source actually reports —
# skip columns that are entirely null so we don't draw empty panels.
import matplotlib.pyplot as plt

SERIES = [
    ("oil_bbl",   "#1a6e3d", "Oil (bbl/mo)"),
    ("gas_mcf",   "#d68b2a", "Gas (mcf/mo)"),
    ("water_bbl", "#2a6cad", "Water (bbl/mo)"),
]
reported = [(col, color, label) for col, color, label in SERIES if prod[col].notna().any()]
omitted  = [label for col, _, label in SERIES if not prod[col].notna().any()]

fig, axes = plt.subplots(len(reported), 1, figsize=(10, 2.3 * len(reported)), sharex=True, squeeze=False)
for ax, (col, color, label) in zip(axes[:, 0], reported):
    ax.fill_between(prod["production_date"], 0, prod[col].fillna(0), color=color, alpha=0.6)
    ax.plot(prod["production_date"], prod[col].fillna(0), color=color, linewidth=0.8)
    ax.set_ylabel(label); ax.grid(alpha=0.2)
axes[0, 0].set_title(f"{well_row['well_name']} — {well_row['operator_name']} ({well_row['external_id']})")
axes[-1, 0].set_xlabel("production month")
plt.tight_layout(); plt.show()

if omitted:
    print(f"Note: {', '.join(omitted)} not reported per-well by this state's regulator.")

In [ ]:
# Rough BOE rollup on whatever volumes are reported.
# BOE = bbl oil + mcf gas / 6.  Water doesn't enter BOE.
prod["boe"] = prod["oil_bbl"].fillna(0) + prod.get("gas_mcf", 0).fillna(0) / 6.0
peak = prod.loc[prod["boe"].idxmax()]
print(f"Cumulative BOE: {prod['boe'].sum():,.0f}")
print(f"Peak month: {peak['production_date'].date()} — {peak['boe']:,.0f} BOE")

## Next steps
- Loop over top-N wells for a single operator and plot combined monthly BOE.
- Pass `as_of=` (Pro+ tier) to replay production as it would have looked on a historical date — useful for back-testing acquisition models.
- Cross with `sondio.us.phmsa.pipeline_incidents(...)` to flag wells feeding into pipelines with recent incidents.
- Related: [earthquake-well-proximity](../cross-vertical/earthquake-well-proximity.ipynb).